# Experimentación: Gráficos de Correlación

Este notebook te permite experimentar con gráficos de dispersión (scatter plots) y correlaciones.

**Instrucciones:**
1. Ejecuta las primeras celdas para cargar datos
2. Ve a la sección "EXPERIMENTA AQUÍ"
3. Cambia solo los parámetros marcados
4. Ejecuta y observa resultados

## 1. Importaciones y Configuración

In [ ]:
# Importaciones.
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Agregar rutas.
Ruta_Base = Path.cwd().parent.parent
sys.path.append(str(Ruta_Base / "Codigo" / "Utilidades"))
sys.path.append(str(Ruta_Base / "Codigo" / "Modelado_Estadistico"))

from Configuracion import *

# Configurar estilo de gráficos.
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

print("✓ Módulos importados correctamente")

## 2. Cargar Datos

In [ ]:
# Cargar datos procesados.
Ruta_Generales = RUTA_BASES_DEFINITIVAS / "Bases finalesGenerales.xlsx"

if Ruta_Generales.exists():
    Df = pd.read_excel(Ruta_Generales)
    print(f"✓ Datos cargados: {len(Df)} filas, {len(Df.columns)} columnas")
else:
    print("⚠️ Archivo no encontrado. Creando datos de ejemplo.")
    np.random.seed(42)
    N = 1000
    Df = pd.DataFrame({
        'Indice_Progresismo': np.random.randn(N) * 0.5 + 3,
        'Indice_Conservadurismo': np.random.randn(N) * 0.5 + 3,
        'Indice_Positividad': np.random.randn(N) * 0.5 + 3,
        'CO_Pro_Izq_Sum': np.random.randn(N) * 2 + 1,
        'CO_Con_Der_Sum': np.random.randn(N) * 2 + 1,
        'CT_Pro_Izq_Sum': np.random.randn(N) * 100 + 50,
        'CT_Con_Der_Sum': np.random.randn(N) * 100 + 50
    })
    print("✓ Usando datos de ejemplo")

print(f"\nPrimeras columnas disponibles:")
print(Df.columns[:20].tolist())

---
# EXPERIMENTA AQUÍ

## Gráfico de Dispersión con Línea de Tendencia

**Cambia estos parámetros:**

In [ ]:
# ============================================================
# PARÁMETROS PARA EXPERIMENTAR
# ============================================================

# Variables para correlación.
Variable_X = 'Indice_Progresismo'  # Variable independiente
Variable_Y = 'CO_Pro_Izq_Sum'      # Variable dependiente

# Título personalizado (None para automático).
Titulo = None

# Color de puntos.
Color_Puntos = 'steelblue'  # Prueba: 'coral', 'green', 'purple'

# Transparencia de puntos (0-1).
Alpha_Puntos = 0.5  # Prueba: 0.3, 0.7, 1.0

# Tamaño de puntos.
Tamano_Puntos = 50  # Prueba: 30, 70, 100

# Color de línea de tendencia.
Color_Linea = 'red'  # Prueba: 'blue', 'green', 'black'

# Ancho de línea.
Ancho_Linea = 2  # Prueba: 1, 3, 4

# Mostrar intervalo de confianza.
Mostrar_Intervalo = True  # Cambia a False para ocultar

# Tamaño de figura.
Tamano_Figura = (10, 6)

# ============================================================
# CALCULAR CORRELACIÓN
# ============================================================

# Eliminar NaN.
Datos_Limpios = Df[[Variable_X, Variable_Y]].dropna()

# Correlación de Spearman.
Rho, P_Valor = stats.spearmanr(
    Datos_Limpios[Variable_X],
    Datos_Limpios[Variable_Y]
)

print(f"\n{'='*60}")
print(f"CORRELACIÓN DE SPEARMAN")
print(f"{'='*60}")
print(f"Variables: {Variable_X} vs {Variable_Y}")
print(f"N: {len(Datos_Limpios)}")
print(f"Rho (ρ): {Rho:.4f}")
print(f"P-valor: {P_Valor:.6f}")
print(
    f"Significativo: {'✓ Sí' if P_Valor < 0.05 else '✗ No'} "
    f"(α = 0.05)"
)
print(f"{'='*60}")

In [ ]:
# ============================================================
# CREAR GRÁFICO DE DISPERSIÓN
# ============================================================

Fig, Ax = plt.subplots(figsize=Tamano_Figura)

# Gráfico de dispersión.
Ax.scatter(
    Datos_Limpios[Variable_X],
    Datos_Limpios[Variable_Y],
    color=Color_Puntos,
    alpha=Alpha_Puntos,
    s=Tamano_Puntos,
    edgecolors='black',
    linewidths=0.5
)

# Línea de tendencia (regresión lineal).
Z = np.polyfit(
    Datos_Limpios[Variable_X],
    Datos_Limpios[Variable_Y],
    1
)
P = np.poly1d(Z)

X_Linea = np.linspace(
    Datos_Limpios[Variable_X].min(),
    Datos_Limpios[Variable_X].max(),
    100
)

Ax.plot(
    X_Linea,
    P(X_Linea),
    color=Color_Linea,
    linewidth=Ancho_Linea,
    label=f'Tendencia lineal'
)

# Intervalo de confianza (opcional).
if Mostrar_Intervalo:
    # Calcular intervalo de confianza 95%.
    from scipy import stats as sp_stats
    
    Slope, Intercept, R_Value, P_Value_Reg, Std_Err = (
        sp_stats.linregress(
            Datos_Limpios[Variable_X],
            Datos_Limpios[Variable_Y]
        )
    )
    
    # Predicciones.
    Y_Pred = Slope * Datos_Limpios[Variable_X] + Intercept
    
    # Error estándar de la predicción.
    N = len(Datos_Limpios)
    Residuos = Datos_Limpios[Variable_Y] - Y_Pred
    S_Res = np.sqrt(np.sum(Residuos**2) / (N - 2))
    
    # Intervalo de confianza.
    T_Val = sp_stats.t.ppf(0.975, N - 2)
    X_Mean = Datos_Limpios[Variable_X].mean()
    Sxx = np.sum((Datos_Limpios[Variable_X] - X_Mean)**2)
    
    Conf_Int = T_Val * S_Res * np.sqrt(
        1/N + (X_Linea - X_Mean)**2 / Sxx
    )
    
    Ax.fill_between(
        X_Linea,
        P(X_Linea) - Conf_Int,
        P(X_Linea) + Conf_Int,
        color=Color_Linea,
        alpha=0.2,
        label='IC 95%'
    )

# Agregar texto con estadísticas.
Texto_Stats = (
    f'ρ = {Rho:.3f}\n'
    f'p = {P_Valor:.4f}\n'
    f'N = {len(Datos_Limpios)}'
)

Ax.text(
    0.05,
    0.95,
    Texto_Stats,
    transform=Ax.transAxes,
    fontsize=11,
    verticalalignment='top',
    bbox=dict(
        boxstyle='round',
        facecolor='white',
        alpha=0.8,
        edgecolor='black'
    )
)

# Configuración de ejes.
Ax.set_xlabel(Variable_X, fontsize=12, fontweight='bold')
Ax.set_ylabel(Variable_Y, fontsize=12, fontweight='bold')

if Titulo:
    Ax.set_title(Titulo, fontsize=14, fontweight='bold')
else:
    Ax.set_title(
        f'Correlación: {Variable_X} vs {Variable_Y}',
        fontsize=14,
        fontweight='bold'
    )

Ax.legend(loc='lower right', frameon=True, shadow=True)
Ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

print(f"\n✓ Gráfico de correlación generado")

## Gráfico de Dispersión con Puntos Coloreados por Categoría

**Cambia estos parámetros:**

In [ ]:
# ============================================================
# PARÁMETROS PARA EXPERIMENTAR
# ============================================================

# Variables.
Variable_X_Cat = 'Indice_Conservadurismo'
Variable_Y_Cat = 'CO_Con_Der_Sum'

# Variable de agrupación (categoría).
Variable_Categoria = 'Categoria_PASO_2023'

# Categorías a incluir (None para todas).
Categorias_Incluir = [
    'Left_Wing',
    'Progressivism',
    'Right_Wing_Libertarian'
]  # O None para todas

# Paleta de colores.
Paleta = 'Set1'  # Prueba: 'Set2', 'Dark2', 'Paired'

# ============================================================
# PREPARAR DATOS
# ============================================================

# Verificar si existe la columna de categoría.
if Variable_Categoria not in Df.columns:
    print(
        f"⚠️ Columna {Variable_Categoria} no encontrada. "
        f"Creando categorías de ejemplo."
    )
    Df[Variable_Categoria] = np.random.choice(
        ['Izquierda', 'Centro', 'Derecha'],
        len(Df)
    )
    Categorias_Incluir = None

# Filtrar por categorías si se especificó.
if Categorias_Incluir:
    Df_Filtrado = Df[
        Df[Variable_Categoria].isin(Categorias_Incluir)
    ].copy()
else:
    Df_Filtrado = Df.copy()

# Eliminar NaN.
Df_Filtrado = Df_Filtrado[
    [Variable_X_Cat, Variable_Y_Cat, Variable_Categoria]
].dropna()

print(f"\n✓ Datos preparados: {len(Df_Filtrado)} casos")
print(f"Categorías: {Df_Filtrado[Variable_Categoria].unique()}")

In [ ]:
# ============================================================
# CREAR GRÁFICO CON CATEGORÍAS
# ============================================================

Fig, Ax = plt.subplots(figsize=(12, 7))

# Crear gráfico con seaborn (más fácil para categorías).
sns.scatterplot(
    data=Df_Filtrado,
    x=Variable_X_Cat,
    y=Variable_Y_Cat,
    hue=Variable_Categoria,
    palette=Paleta,
    s=80,
    alpha=0.6,
    edgecolor='black',
    linewidth=0.5,
    ax=Ax
)

# Líneas de tendencia por categoría.
Categorias_Unicas = Df_Filtrado[Variable_Categoria].unique()
Colores = sns.color_palette(Paleta, len(Categorias_Unicas))

for i, Cat in enumerate(Categorias_Unicas):
    Datos_Cat = Df_Filtrado[
        Df_Filtrado[Variable_Categoria] == Cat
    ]
    
    if len(Datos_Cat) > 1:
        # Regresión lineal.
        Z_Cat = np.polyfit(
            Datos_Cat[Variable_X_Cat],
            Datos_Cat[Variable_Y_Cat],
            1
        )
        P_Cat = np.poly1d(Z_Cat)
        
        X_Cat_Linea = np.linspace(
            Datos_Cat[Variable_X_Cat].min(),
            Datos_Cat[Variable_X_Cat].max(),
            100
        )
        
        Ax.plot(
            X_Cat_Linea,
            P_Cat(X_Cat_Linea),
            color=Colores[i],
            linewidth=2,
            linestyle='--',
            alpha=0.8
        )

# Configuración.
Ax.set_xlabel(Variable_X_Cat, fontsize=12, fontweight='bold')
Ax.set_ylabel(Variable_Y_Cat, fontsize=12, fontweight='bold')
Ax.set_title(
    f'Correlación por {Variable_Categoria}: '
    f'{Variable_X_Cat} vs {Variable_Y_Cat}',
    fontsize=14,
    fontweight='bold'
)
Ax.legend(title=Variable_Categoria, loc='best', frameon=True, shadow=True)
Ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

print(f"\n✓ Gráfico por categorías generado")

## Matriz de Correlaciones (Heatmap)

**Cambia estos parámetros:**

In [ ]:
# ============================================================
# PARÁMETROS PARA EXPERIMENTAR
# ============================================================

# Variables a incluir en la matriz.
Variables_Matriz = [
    'Indice_Progresismo',
    'Indice_Conservadurismo',
    'Indice_Positividad',
    'CO_Pro_Izq_Sum',
    'CO_Con_Der_Sum'
]

# Paleta de colores.
Paleta_Heatmap = 'coolwarm'  # Prueba: 'RdBu_r', 'viridis', 'rocket'

# Mostrar valores de correlación.
Mostrar_Valores = True

# Formato de valores (decimales).
Formato_Valores = '.2f'  # Prueba: '.3f', '.1f'

# Tamaño de figura.
Tamano_Heatmap = (10, 8)

# ============================================================
# CALCULAR MATRIZ DE CORRELACIONES
# ============================================================

# Filtrar variables que existen en el DataFrame.
Variables_Existentes = [
    V for V in Variables_Matriz if V in Df.columns
]

if len(Variables_Existentes) < 2:
    print(
        f"⚠️ Menos de 2 variables encontradas. "
        f"Usando todas las columnas numéricas."
    )
    Variables_Existentes = Df.select_dtypes(
        include=[np.number]
    ).columns[:10].tolist()

# Calcular correlación de Spearman.
Matriz_Corr = Df[Variables_Existentes].corr(method='spearman')

print(f"\n✓ Matriz de correlaciones calculada")
print(f"Dimensiones: {Matriz_Corr.shape}")
print(f"\nMatriz:")
print(Matriz_Corr)

In [ ]:
# ============================================================
# CREAR HEATMAP
# ============================================================

Fig, Ax = plt.subplots(figsize=Tamano_Heatmap)

# Crear heatmap.
sns.heatmap(
    Matriz_Corr,
    annot=Mostrar_Valores,
    fmt=Formato_Valores,
    cmap=Paleta_Heatmap,
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8},
    ax=Ax
)

# Configuración.
Ax.set_title(
    'Matriz de Correlaciones (Spearman)',
    fontsize=14,
    fontweight='bold',
    pad=20
)

plt.tight_layout()
plt.show()

print(f"\n✓ Heatmap generado")

---
## Guardar Gráfico

Cuando tengas un gráfico que te guste, guárdalo así:

In [ ]:
# Definir ruta de salida.
Ruta_Guardado = (
    RUTA_GRAFICOS /
    "Correlaciones" /
    "Mi_Correlacion.png"
)

# Crear carpeta si no existe.
Ruta_Guardado.parent.mkdir(parents=True, exist_ok=True)

# Guardar el gráfico actual.
Fig.savefig(
    Ruta_Guardado,
    dpi=300,
    bbox_inches='tight'
)

print(f"✓ Gráfico guardado en: {Ruta_Guardado}")